# Config and Imports

In [ ]:
import torch
import torch.nn as nn
import os
from PIL import Image
from torchvision import transforms
#import rasterio
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import normalize
import time
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage import color
import matplotlib as mpl
import random
import albumentations as A
import pandas as pd
import math
import seaborn as sn
from datetime import datetime
from enum import Enum
import copy

#paths to folders containing patches for each modality, training and testing patches are stored in seperate folders

MASKS_PATH = ...
RGB_IMAGES_PATH = ...
LMS_IMAGES_PATH = ...
HSI_IMAGES_PATH= ...

#use gpu when avaialable
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


LMS_CHANNELS = 3
RGB_CHANNELS = 3
HSI_CHANNELS = 48
NUM_CLASSES = 20
INPUT_IMAGE_WIDTH = 149
INPUT_IMAGE_HEIGHT = 601

#hyperparams
INIT_LR = 0.005
NUM_EPOCHS = 15
BATCH_SIZE = 4


CLASS_NAMES = ["Healthy grass", "Stressed grass", "Artificial turf", "Evergreen trees", "Deciduous trees", "Bare earth", "Water", "Residential buildings", "Non-residential buildings", "Roads", "Sidewalks", "Crosswalks", "Major thoroughfares", "Highways", "Railways", "Paved parking lots", "Unpaved parking lots", "Cars", "Trains", "Stadium seats","Unclassified"]


print(f"Using device: {DEVICE}")
torch.manual_seed(42)
np.random.seed(42)

BASE_OUTPUT = ''
MODELS_PATH = ''
PREDICTIONS_PATH = ''
UNCERTAINTIES_PATH = ''

In [ ]:
class FusionType(Enum):
    EARLY = 1
    MIDDLE = 2
    LATE = 3
    NO_FUSION = 4
    

# Define Models

## Unet Parts

In [ ]:
#parts of the unet architecture

#double convolution with batchNorm and dropout
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, drop_channels=True,p=0.25):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(in_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels),
        )
        if drop_channels:
           self.double_conv.append(nn.Dropout2d(p=p))

    def forward(self, x):
        return self.double_conv(x)

#one part of the contracting path, double convolution and maxpool
class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)
    
    
#one part of the expanding path, transposed convolution to upsample the input and double convolution  
class Up(nn.Module):

    def __init__(self, in_channels, out_channels,output_padding= (0,0)):
        super().__init__()
        #output padding is used to account for the odd shape of the patches (601x149) in the DFC dataset
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2,output_padding = output_padding)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1,x2):
        x1 = self.up(x1)
        
        #input is upsampled and concatenated with the skip connection (x2)
        return self.conv(torch.cat((x1,x2),1))

## Basic Unet - No Fusion or Early Fusion

In [ ]:
#basic unet with no fusion or early fusion
class UNet_basic(nn.Module):
    def __init__(self, n_classes, n_channels):
        super(UNet_basic, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        self.inc = DoubleConv(n_channels, 64, drop_channels=False)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        #output padding is used to account for the odd shape of the patches (601x149)
        self.up1 = Up(1024, 512,output_padding = (1,0))
        self.up2 = Up(512, 256,output_padding = (0,1))
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64,output_padding = (1,1))
        self.outc = nn.Conv2d(64, n_classes,kernel_size=1)

    def forward(self, xs):
        x = xs[0]
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5,x4)
        x = self.up2(x,x3)
        x = self.up3(x,x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        #return logits as the cross-entropy loss function applies softmax
        return logits

## Unet with middle fusion

In [ ]:
#unet model implementing middle fusion
class UNet_middle(nn.Module):
    def __init__(self, n_classes, n_channels):
        super(UNet_middle, self).__init__()
        
        #n_channels is a list with number of channels for each modality
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.num_modalities = len(n_channels)
        
        #each modality has its own contracting path
        self.incs = torch.nn.ModuleList([DoubleConv(channels, 64, drop_channels=False) for channels in self.n_channels])
        self.downs1 = torch.nn.ModuleList([Down(64, 128) for channels in self.n_channels])
        self.downs2 = torch.nn.ModuleList([Down(128, 256) for channels in self.n_channels])
        self.downs3 = torch.nn.ModuleList([Down(256, 512) for channels in self.n_channels])
        self.downs4 = torch.nn.ModuleList([Down(512, 1024) for channels in self.n_channels])
        
        #number of channels is multiplied by num_modalities to match the amount of channels in the fused skip connections
        self.up1 = Up(1024*self.num_modalities, 512*self.num_modalities,output_padding = (1,0))
        self.up2 = Up(512*self.num_modalities, 256*self.num_modalities,output_padding = (0,1))
        self.up3 = Up(256*self.num_modalities, 128*self.num_modalities)
        self.up4 = Up(128*self.num_modalities, 64*self.num_modalities,output_padding = (1,1))
        self.outc = nn.Conv2d(64*self.num_modalities, n_classes,kernel_size=1)

    def forward(self, xs):
        x1,x2,x3,x4,x5 = [],[],[],[],[]
        
        # execute the contracting path for each modality
        for i in range(len(xs)):
            x1.append(self.incs[i](xs[i]))
            x2.append(self.downs1[i](x1[i]))
            x3.append(self.downs2[i](x2[i]))
            x4.append(self.downs3[i](x3[i]))
            x5.append(self.downs4[i](x4[i]))

        #x5 and the skip connections are concatenated (shape is BxCxHxW, concat along C axis)
        x = self.up1(torch.cat(x5,1), torch.cat(x4,1))
        x = self.up2(x, torch.cat(x3,1))
        x = self.up3(x, torch.cat(x2,1))
        x = self.up4(x, torch.cat(x1,1))

        logits = self.outc(x)

        return logits

## Unet with late fusion

In [ ]:

#simple late fusion model, the softmax outputs from each modality is averaged
class UNet_late_simple(nn.Module):
    def __init__(self, n_classes, models):
        super(UNet_late_simple, self).__init__()
        self.models = nn.ModuleDict([
                [m.name, copy.deepcopy(m.model)] for m in models
        ])
        #weights should not be updated
        for model in self.models:
            for param in self.models[model].parameters():
                param.requires_grad = False
        
        self.n_classes = n_classes
        self.mcdo = False
        

    def forward(self, xs):
        #Shape B,C,H,W
        #Averege along C Axis
        
        #every model has to be set to eval seperately
        
        #get softmax from first model
        first_model = self.models[list(self.models.keys())[0]]
        first_model.eval()
        
        #if mcdo = true activate dropout
        if self.mcdo:
            for m in first_model.modules():
                if m.__class__.__name__.startswith('Dropout'):
                    m.train()
        y_final = first_model([xs[0]])
        y_final = nn.functional.softmax(y_final,dim=1)
        
        #get predictions from other models
        for i,model in enumerate(self.models):
            if i == 0:
                continue
            self.models[model].eval()
            if self.mcdo:
                for m in self.models[model].modules():
                    if m.__class__.__name__.startswith('Dropout'):
                        m.train()
            
            y = self.models[model]([xs[i]])
            y = nn.functional.softmax(y,dim=1)
            y_final = y_final + y
        y_final = y_final / len(self.models.keys())
        return y_final
        

In [ ]:
#late fusion with a linear layer to fuse the logit outputs  from the modalities
class UNet_late(nn.Module):
    def __init__(self, n_classes, models ):
        super(UNet_late, self).__init__()
        
        self.models = nn.ModuleDict([
                [m.name, copy.deepcopy(m.model)] for m in models
        ])
        #weights should not be updated
        for model in self.models:
            for param in self.models[model].parameters():
                param.requires_grad = False
        self.n_classes = n_classes
        
        #has to be true during mcdo
        self.mcdo = False
        
        #linear layer
        self.outc = nn.Conv2d(self.n_classes*len(self.models), n_classes,kernel_size=1)

    def forward(self, xs):
        #Shape B,C,H,W
        
        logits = []
        for i,model in enumerate(self.models):
            
            #every model has to be set to eval seperately
            self.models[model].eval()
            if self.mcdo:
                for m in self.models[model].modules():
                    if m.__class__.__name__.startswith('Dropout'):
                        m.train()
            logits.append(self.models[model]([xs[i]]))
        #concat along C Axis
        y = torch.cat(logits,dim=1)
        y = self.outc(y)
        return y

# Datasets

## Universal Dataset

In [ ]:
class LanduseDataset(torch.utils.data.Dataset):

    def __init__(self, image_paths:list, mask_paths, fusion_type:FusionType, augs = None):
         #list with paths for all modalities
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.fusion_type = fusion_type
        #transform has to be image only
        self.transform = augs if augs is None else A.Compose(augs)

    
    def __getitem__(self, i):
        #load all images
        images = [np.load(image_path[i]).astype(np.float32) for image_path in self.image_paths] 
        #images are in shape HxWxC
        #concatenate the images if fusion type is early, concat along C axis
        if(self.fusion_type == FusionType.EARLY):
            images = [np.concatenate(images,2)]
            
        mask = np.load(self.mask_paths[i]).astype(np.int16)
        
        if self.transform is not None:
           
            images = [self.transform(image = img)['image'] for img in images]
        # unclassified should be -1, it is 0 in the original data
        mask = mask - 1
        #trasnpose to shape CxHxW, as required for pytorch
        images = [torch.from_numpy(image.transpose((2,0,1))).type(torch.FloatTensor) for image in images]
        mask = torch.from_numpy(mask.transpose((2,0,1))).squeeze().type(torch.LongTensor)
        #images is a list, in case of early or no fusion with only one element
        return images, mask

    def __len__(self):
        return len(self.mask_paths)

    def test_time_augment(self, i, augs_image_only:list,geometric_augs:list):
        aug_images = A.Compose(augs_image_only)
        #has to be replay to use the same params for every modality. if list of images is passed to an augmentation the paraemters are randomized
        
        geo_aug = A.ReplayCompose(geometric_augs)
        
        images = [np.load(image_path[i]).astype(np.float32) for image_path in self.image_paths] 
        
        if(self.fusion_type == FusionType.EARLY):
            images = [np.concatenate(images,0)]
            
        mask = np.load(self.mask_paths[i])
        #unclassified is -1, this doeasn't work with albumentations
        mask = np.vectorize(lambda x: x if x >= 0 else NUM_CLASSES)(mask)
        #needs to be done seperately since modalities have different shapes
        #transpose to HxWxC for augs, albumentations format
        images = [img.transpose(1,2,0) for img in images]
        #apply the augmentation to the first modality and mask and save the params
        images = [aug_images(image = img)['image'] for img in images]
        
        fully_augmented = geo_aug( image = images[0],mask = mask)
        mask = fully_augmented['mask']
        geo_aug_params = fully_augmented['replay']
        
         #replay the aug for all imgs
        images = [A.ReplayCompose.replay(geo_aug_params, image = img)["image"] for img in images]
        
        images = [img.transpose(2,0,1) for img in images]

        
        images = [torch.from_numpy(image).type(torch.FloatTensor) for image in images]
        mask = torch.from_numpy(mask).type(torch.LongTensor)

        return images, mask, geo_aug_params
    

# Training

## Initialize models and datasets

### Define some configs

In [ ]:
#wrapper class for a model
class ModelWrapper:
    def __init__(self, name, unet, train_loader, test_loader, train_dataset, test_dataset, optimizer,criterion,fusion_type):
        self.name = name
        self.model = unet
        self.trainLoader = train_loader
        self.testLoader = test_loader
        self.optimizer = optimizer
        #loss function
        self.criterion = criterion
        self.trainDataset = train_dataset
        self.testDataset = test_dataset
        self.H = {"train_loss": [], "test_loss": []}
        self.fusion_type = fusion_type
        self.lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer,factor = 0.5, patience = 30) if self.optimizer is not None else None
        #dictionary to save uncertainty maps, pred-> agreement uncertainty, soft-> predictive uncertainty (calculated with softmax)
        self.uncertainty_dict = {
                            "tta":{"uncertainties_pred":[], "uncertainties_soft":[]},
                            "mcdo":{"uncertainties_pred":[], "uncertainties_soft":[]}
                            }
        

In [ ]:

#function to create models, parameter models is the list of models for late fusion
def create_model(name, modalities, channels_dict, paths_dict, fusion_type= FusionType.NO_FUSION, aug_pipe = None, models = None):
    
    num_workers = 0
    if 'hsi' in modalities:
        num_workers = 4
        
    training_data = [sorted([paths_dict[mod] + f"train/{fp}" for fp in os.listdir(paths_dict[mod] + "train/")]) for mod in modalities]
    training_masks = sorted([MASKS_PATH+"train/"+fp for fp in os.listdir(MASKS_PATH+"train")])
    testing_data = [sorted([paths_dict[mod] + f"test/{fp}" for fp in os.listdir(paths_dict[mod] + "test/")]) for mod in modalities]
    testing_masks = sorted([MASKS_PATH+"test/"+fp for fp in os.listdir(MASKS_PATH+"test")])

    trainDataset = LanduseDataset(
        training_data,
        training_masks,
        fusion_type,
        aug_pipe
    )

    testDataset = LanduseDataset(
        testing_data,
        testing_masks,
        fusion_type
    )
    trainDataloader = torch.utils.data.DataLoader(trainDataset, shuffle=True, batch_size=BATCH_SIZE, num_workers = num_workers)
    testDataloader = torch.utils.data.DataLoader(testDataset, shuffle=False, batch_size=BATCH_SIZE, num_workers = num_workers)

    if fusion_type == FusionType.EARLY or fusion_type == FusionType.NO_FUSION:
        model = UNet_basic(NUM_CLASSES,sum([channels_dict[mod] for mod in modalities]))
    elif fusion_type == FusionType.MIDDLE:
        model = UNet_middle(NUM_CLASSES,[channels_dict[mod] for mod in modalities])
    elif fusion_type == FusionType.LATE:
        assert models is not None
        model = UNet_late(NUM_CLASSES,models)
    elif fusion_type == FusionType.LATE_SIMPLE:
        assert models is not None
        model = UNet_late_simple(NUM_CLASSES,models)
        
        
    return ModelWrapper(
        name=name,
        unet=model,
        train_loader=trainDataloader,
        test_loader=testDataloader,
        train_dataset=trainDataset,
        test_dataset=testDataset,
        criterion = torch.nn.CrossEntropyLoss(ignore_index=-1),
         #late simple does not have any trainable parameters
        optimizer=torch.optim.Adam(model.parameters(), lr=INIT_LR) #if fusion_type != FusionType.LATE_SIMPLE else None
        ,fusion_type=fusion_type
    )

In [ ]:
modalities = ['hsi','lms','rgb']
all_possible_combinations = [["hsi","lms"],["hsi","rgb"],["lms","rgb"],["hsi","lms","rgb"]]


#list with strings of image paths
test_image_paths = {
    "mask" : [os.path.join(MASKS_PATH,'test/', image_id) for image_id in sorted(os.listdir(MASKS_PATH+'test/'))],
    "rgb":[os.path.join(RGB_IMAGES_PATH,'test/', image_id) for image_id in sorted(os.listdir(RGB_IMAGES_PATH+'test/'))],
    "lms":[os.path.join(LMS_IMAGES_PATH,'test/', image_id) for image_id in sorted(os.listdir(LMS_IMAGES_PATH+'test/'))],
    "hsi" :[os.path.join(HSI_IMAGES_PATH,'test/', image_id) for image_id in sorted(os.listdir(HSI_IMAGES_PATH+'test/'))]
}

train_image_paths = {
    "mask" : [os.path.join(MASKS_PATH,'train/', image_id) for image_id in sorted(os.listdir(MASKS_PATH+'train/'))],
    "rgb":[os.path.join(RGB_IMAGES_PATH,'train/', image_id) for image_id in sorted(os.listdir(RGB_IMAGES_PATH+'train/'))],
    "lms":[os.path.join(LMS_IMAGES_PATH,'train/', image_id) for image_id in sorted(os.listdir(LMS_IMAGES_PATH+'train/'))],
    "hsi" :[os.path.join(HSI_IMAGES_PATH,'train/', image_id) for image_id in sorted(os.listdir(HSI_IMAGES_PATH+'train/'))]
}


modalities_paths = {
    "lms": LMS_IMAGES_PATH,
    "hsi": HSI_IMAGES_PATH,
    "mask":MASKS_PATH,
    "rgb":RGB_IMAGES_PATH
}

modalities_channels = {
    "hsi": HSI_CHANNELS,
    "lms": LMS_CHANNELS,
    "rgb":RGB_CHANNELS
}

models = {}

datasets = {}
dataloaders = {}

trainSteps = len(train_image_paths['mask']) // BATCH_SIZE
testSteps = len(test_image_paths['mask']) // BATCH_SIZE
#augmentations for training
aug_pipe = [
    A.GaussNoise(p=0.5,  mean=0.0,var_limit=(0.000005,0.00001),per_channel=True)
]

models={
    FusionType.NO_FUSION:{},
    FusionType.EARLY:{},
    FusionType.MIDDLE:{},
    FusionType.LATE:{}
}


fusion_types = [FusionType.EARLY,FusionType.MIDDLE, FusionType.LATE]

for mod in modalities:
    name = mod
    models[FusionType.NO_FUSION][name] = create_model(
        name, [mod], modalities_channels, modalities_paths, FusionType.NO_FUSION, aug_pipe
    )
for fusion in fusion_types:
    for combo in all_possible_combinations:
        name = "_".join(combo)
        baselines = None
        
        if fusion == FusionType.LATE:
            #unimodal networks for late fusion
            baselines =[models[FusionType.NO_FUSION][model_name] for model_name in combo]

        models[fusion][name] = create_model(
            name, combo, modalities_channels, modalities_paths, fusion, aug_pipe,baselines
        )


## Define Training Loop

In [ ]:
#training loop
def train_model(model):
    
    
    print("[INFO] training the network...")
    
    startTime = time.time()
    network = model.model
    opt = model.optimizer
    scheduler = model.lr_scheduler
    lossFunc = model.criterion
    network.to(DEVICE)
    
    #to average loss
    trainSteps = len(model.trainDataset) // BATCH_SIZE
    testSteps = len(model.testDataset) // BATCH_SIZE
    
    for e in tqdm(range(NUM_EPOCHS)):
        network.train()
        
        totalTrainLoss = 0
        totalTestLoss = 0
        
        for (i, (image_list, mask)) in enumerate(model.trainLoader):
            #image_list is an array containing different modalities, even for one modality
            (image_list, mask) = ([mod.to(DEVICE) for mod in image_list], mask.to(DEVICE))
            pred = network(image_list)
            loss = lossFunc(pred, mask)
            opt.zero_grad(set_to_none = True)
            #backprop
            loss.backward()
            opt.step()
            totalTrainLoss += loss
        if scheduler is not None:
            scheduler.step(totalTrainLoss)
    
        with torch.no_grad():
        
            network.eval()
            
            for (image_list, mask) in model.testLoader:
            
                (image_list, mask) = ([mod.to(DEVICE) for mod in image_list], mask.to(DEVICE))
                pred = network(image_list)
                totalTestLoss += lossFunc(pred, mask)
                
            
        avgTrainLoss = totalTrainLoss / trainSteps
        avgTestLoss = totalTestLoss / testSteps
        if scheduler is not None:
            scheduler.step(totalTestLoss) 
            
        model.H["train_loss"].append(avgTrainLoss.cpu().detach().numpy())
        model.H["test_loss"].append(avgTestLoss.cpu().detach().numpy())
                
        print("[INFO] EPOCH: {}/{}".format(e + 1, NUM_EPOCHS))
        print("Train loss: {:.6f}, Test loss: {:.4f}".format(avgTrainLoss, avgTestLoss))
    
    endTime = time.time()
    print("[INFO] total time taken to train the model: {:.2f}s".format(endTime - startTime))

## Load models

In [ ]:
#function used to load the model weights. naming convention is path_fusion__name.pth
def load_model(model,models_path):
    name = model.name
    #print(name)
    fusion = "no"
    if model.fusion_type == FusionType.EARLY:
        fusion = "early"
    elif model.fusion_type == FusionType.MIDDLE:
        fusion = "mid"
    elif model.fusion_type == FusionType.LATE:
        fusion = "late"
    elif model.fusion_type == FusionType.LATE_SIMPLE:
        return

    if len(model.name) == 3 + 3*4:
        name = "all"
    path = models_path + fusion +"_" + name + ".pth"
    state_dict = torch.load(path, weights_only=True,map_location=torch.device('cpu'))
    #only the linear layer of the late linear models was saved, late linear models should be initialized and loaded after unimodal networks have been loaded
    if model.fusion_type == FusionType.LATE:
        with torch.no_grad():
            model.model.outc.weight.copy_(state_dict['outc.weight'])
            model.model.outc.bias.copy_(state_dict['outc.bias'])
    else:
        model.model.load_state_dict(state_dict)
    model.model.to(DEVICE)
    print(model.name + " " + str(model.fusion_type) + " loaded_successfully!")

In [ ]:
for fusion in models.keys():
    for key in models[fusion].keys():

        load_model(models[fusion][key],MODELS_PATH)

#create simple fusion models, once the unimodal networks have been loaded
for combo in all_possible_combinations:
        fusion = FusionType.LATE_SIMPLE
        name = "_".join(combo)
        print(name,end = ",")
        
        baselines =[models[FusionType.NO_FUSION][model_name] for model_name in combo]

        models[fusion][name] = create_model(
            name, combo, modalities_channels, modalities_paths, fusion, aug_pipe,baselines
        )

        



## Training...

In [ ]:
current_model = models[FusionType.MIDDLE]['lms_rgb']
train_model(current_model)

In [ ]:

plt.figure()
plt.plot(current_model.H["train_loss"], label="train_loss")
plt.plot(current_model.H["test_loss"], label="test_loss")
#plt.title(f"Training {current_model.name}")
plt.xlabel("Epoch #")
plt.ylabel("Loss")
plt.legend(loc="lower left")
plt.savefig(os.path.join(BASE_OUTPUT, str(input("Enter plot name:"))+ ".png"))

# serialize the model to disk
torch.save(current_model.model.state_dict(), os.path.join(BASE_OUTPUT, str(input("Enter model name:"))+ ".pth"))

# Evaluate 

## Define functions for plotting

In [ ]:
#functions for showing the hsi, lms and rgb dont need this, since they have 3 channels
   
def show_hsi(arr):
    z = np.empty((3,arr.shape[0],arr.shape[1]))
    #get the rgb channels
    z[0],z[1],z[2] = arr[:,:,3],arr[:,:,2],arr[:,:,1]
    z = z.transpose(1,2,0)
    norm_and_sharp = A.Compose(
        [A.Normalize(normalization = 'min_max_per_channel')
        ]
    )
    return norm_and_sharp(image = z)['image']



In [ ]:
from matplotlib import colors

def prepare_plot(origImage, origMask, predMask, img_type = 'rgb'):

    if img_type == 'hsi':
        origImage = show_hsi(origImage)
        
    #legend with class names, correct colors per class
    ticks = [i + 0.5 for i in range(0,NUM_CLASSES+1)]
    norm = colors.BoundaryNorm([x-0.5 for x in ticks]+[NUM_CLASSES+1],NUM_CLASSES+1)
    fig, ax = plt.subplots(1, 3)
    mask_colors = mpl.colormaps['tab20'].colors
    cmap = colors.LinearSegmentedColormap.from_list("cmap", mask_colors[:NUM_CLASSES+3], N=NUM_CLASSES+1)
    
     #interpolation = nearest to turn off anti aliasing
    ax[0].imshow(origImage,norm = 'linear',interpolation = "nearest")
    gt = ax[1].imshow(origMask,cmap = cmap,norm=norm,interpolation = "nearest")
    ax[2].imshow(predMask,cmap = cmap,norm=norm,interpolation = "nearest")
    
    ax[0].set_title("Image")
    ax[1].set_title("Original Mask")
    ax[2].set_title("Predicted Mask")
    
    #legend with class labels and colors
    cbar = fig.colorbar(gt, ax=ax, orientation='horizontal',fraction=.05,ticks = ticks)
    cbar.ax.set_xticklabels(CLASS_NAMES)
    cbar.ax.xaxis.set_tick_params(rotation=90, labelsize=10)
  
    plt.show()

In [ ]:
#segment a given testing image with a given model
#returns the image of the first modality, original mask and segmentation
def make_predictions(model_dict, id, show_plot = True, img_type = 'rgb'):
    model = model_dict.model
    model.eval()
    
    with torch.no_grad():
        
        images,mask = model_dict.testDataset[id]
        images = [im.to(DEVICE).unsqueeze(0) for im in images]
        predMask = model(images).squeeze()
        
        #transform from logits to a single class id
        predMask = torch.argmax(predMask, dim=0)
        
        #prediction is in shape BxCxHxW, squeeze gets rid of the first dimension, transpose to shape needed for pyplot
        image = images[0].squeeze(0).cpu().numpy().transpose((1,2,0))

        
        mask =mask.cpu().numpy()

        mask = np.vectorize(lambda x: x if x >= 0 else NUM_CLASSES)(mask)
        
        predMask = predMask.cpu().numpy()
        
        predMask = predMask.astype(np.uint8)
        if show_plot:
            prepare_plot(image, mask, predMask,img_type)
    return (image,mask,predMask)  


In [ ]:
current_model = models[FusionType.LATE]['lms_rgb']
current_model.model.to(DEVICE)

x = random.randint(0,len(current_model.testDataset))
image,mask,predMask = make_predictions(current_model, x,img_type = current_model.name[3:])


In [ ]:
#get number of samples for each class
def get_sample_class_numbers(models):
    trains = [0 for _ in CLASS_NAMES]
    tests = [0 for _ in CLASS_NAMES]


    for train_mask in models[FusionType.NO_FUSION]['rgb'].trainDataset:
        mask = train_mask[1].numpy()
        z = np.unique(mask,return_counts=True)
        for i in range(len(z[0])):
            trains[z[0][i]] += z[1][i]

    for train_mask in models[FusionType.NO_FUSION]['rgb'].testDataset:
        mask = train_mask[1].numpy()
        z = np.unique(mask,return_counts=True)
        for i in range(len(z[0])):
            tests[z[0][i]] += z[1][i]
   
    return (trains,tests)


In [ ]:
trains,tests = get_sample_class_numbers(models)

In [ ]:
def plot_class_samples(trains,tests):
    xs = [i+1 for i in range(len(trains))]
    colors = mpl.colormaps['tab20'].colors
    labels = CLASS_NAMES[:-1]
    width = 0.33
    
    for i in range(len(trains)):
        plt.bar([x-width/2 -0.02 for x in xs],trains,width = width,color = colors)
        plt.bar([x+width/2+0.02 for x in xs],tests,width = width,color = colors)
    plt.yscale('log')
    plt.xticks(ticks=xs,labels=labels,rotation=90)
    plt.savefig("dfc_class_samples.pdf",bbox_inches="tight")
    plt.show()
    

In [ ]:
plot_class_samples(trains[:-1],tests[:-1])

## Calculating accuracy metrics

In [ ]:
#guesses contains a list for each class, with entry i containing the number of times where a given class was predicted while class i was correct
#calculate Cohen's kappa
def calculate_kappa(OA,guesses):
    
    guesses = np.array(guesses)
    tot = np.sum(guesses)
    p= sum([(np.sum(guesses[i])/tot)*(np.sum(guesses.T[i])/tot) for i in range(len(guesses))] )

    return (OA- p)/(1-p)

#calculate the accuracy metrics OA,AA,mIOU and k
def evaluate_model(current_model:ModelWrapper):
    model = current_model.model
    data = current_model.testDataset
    model.eval()
    #first row contains the number of times where each class was predicted, while class 0 was correct etc.
    guesses_per_class = [[0 for _ in range(NUM_CLASSES)] for _ in range(NUM_CLASSES)]

    with torch.no_grad():
        for img_id in range(len(data)):
            _,mask,prediction = make_predictions(current_model,img_id,False)
            for i in range(INPUT_IMAGE_HEIGHT):
                for j in range(INPUT_IMAGE_WIDTH):
                    if mask[i][j] != NUM_CLASSES:
                        
                        guesses_per_class[mask[i][j]][prediction[i][j]] += 1
    #transform to frequencies
    confusion_matrix = list(map(lambda x: list(map(lambda y: round(y/sum(x),2))),guesses_per_class))
    #data frame for seaborn
    cm_df = pd.DataFrame(confusion_matrix,index = [cls for cls in CLASS_NAMES[:-1]],columns = [cls for cls in CLASS_NAMES[:-1]])
    
    #show confusion matrix
    plt.figure(figsize = (10,7))
    sn.heatmap(cm_df, annot=True)
    
    #diagonal
    accuracy_per_class = [confusion_matrix[i][i] for i in range(len(confusion_matrix))]
    #sum of number of correct pixels for each class divided by the total number of labeled pixels
    overall_acc = sum([accuracy_per_class[i]*sum(guesses_per_class[i]) for i in range(len(accuracy_per_class))])/sum([sum(x) for x in guesses_per_class])
    
    print(f"Averge accuracy: {sum(accuracy_per_class)*100/len(accuracy_per_class):.2f}% ")
    print(f"Overall accuracy: {overall_acc*100:.2f}%")
    ious = []
    for i in range(NUM_CLASSES):
        iou = guesses_per_class[i][i]/(sum(guesses_per_class[i]) + sum(np.array(guesses_per_class).T[i]) - guesses_per_class[i][i]) if sum(guesses_per_class[i]) >0 else 0
        ious.append(iou)
    print(f"mIOU: {sum(ious)*100/len(ious):.2f}%")
    print(f"Kappa: {calculate_kappa(overall_acc,guesses_per_class)}")

   

In [ ]:
#same function as before, without confusion matrix and print statements, returns the performance metrics
def get_metrics(current_model):
    results = {
        "classes":[],
        "AA":0,
        "OA":0,
        "mIOU":0,
        "k":0
    }
    
    model = current_model.model
    data = current_model.testDataset
    model.eval()
    guesses_per_class = [[0 for _ in range(NUM_CLASSES)] for _ in range(NUM_CLASSES)]
    with torch.no_grad():
        for img_id in range(len(data)):
            _,mask,prediction = make_predictions(current_model,img_id,False)
            for i in range(INPUT_IMAGE_HEIGHT):
                for j in range(INPUT_IMAGE_WIDTH ):
                    if mask[i][j] != NUM_CLASSES:
                        try:
                            guesses_per_class[mask[i][j]][prediction[i][j]] += 1
                        except IndexError:
                            print(img_id, mask[i][j], prediction[i][j])
    
    confusion_matrix = list(map(lambda x: list(map(lambda y: round(y/sum(x),2) if sum(x) > 0 else y,x)),guesses_per_class))
    
    accuracy_per_class = [confusion_matrix[i][i] for i in range(len(confusion_matrix))]
    results["classes"] = accuracy_per_class
    results["AA"] = sum(accuracy_per_class)/len(accuracy_per_class)

    overall_acc = sum([accuracy_per_class[i]*sum(guesses_per_class[i]) for i in range(len(accuracy_per_class))])/sum([sum(x) for x in guesses_per_class])
    results["OA"] =   overall_acc
    ious = []
    for i in range(NUM_CLASSES):
        iou = guesses_per_class[i][i]/(sum(guesses_per_class[i]) + sum(np.array(guesses_per_class).T[i]) - guesses_per_class[i][i]) if sum(guesses_per_class[i]) >0 else 0
        ious.append(iou)
    results["mIOU"] = sum(ious)/len(ious)
    results["k"] = calculate_kappa(overall_acc,guesses_per_class)
    return results


In [ ]:
#get metrics for all models
def get_all_results_to_string(models):
    final_string = "fusion_type;modalities;" + ";".join(CLASS_NAMES[0:-1])+";OA;AA;mIOU;k\n"
    for fusion_type in models.keys():
        for model_key in models[fusion_type].keys():
            print("Fetching results for " + str(fusion_type) + " " + model_key)
            model = models[fusion_type][model_key]
            
            model_string = str(model.fusion_type).split(".")[1] + ";" + model.name + ";"
            
            metrics = get_metrics(model)
            model.metrics = metrics
            model_string += ";".join( [str(acc) for acc in metrics['classes']]) + ";" 
            model_string += str(metrics["OA"]) + ";"
            model_string += str(metrics["AA"]) + ";"
            model_string += str(metrics["mIOU"]) + ";"
            model_string += str(metrics["k"])+"\n"

            final_string +=model_string
    return final_string

In [ ]:
#save all predictions
def get_preds(current_model):
    results = {
        "masks":[0 ],
        "preds":[]
    }
    
    model = current_model.model
    data = current_model.testDataset
    results = {
        "masks":[0 for _ in data ],
        "preds":[0 for _ in data ]
    }
    model.eval()
    with torch.no_grad():
        for img_id in range(len(data)):
            _,mask,prediction = make_predictions(current_model,img_id,False)
            results['masks'][img_id] = mask
            results['preds'][img_id] = prediction
        name = PREDICTIONS_PATH + str(current_model.fusion_type).split(".")[1]+"_"+current_model.name + "_predictions.pth"
        np.save(PREDICTIONS_PATH + name,results)
    print(name + " saved!")
    


In [ ]:
#apply a given function to all models
def apply_fn_to_all(models, fn):
    for fusion in models.keys():
        for key in models[fusion].keys():
            fn(models[fusion][key])


## Calculating uncertainty

In [ ]:
# function for plotting mask, segmentation and uncertainty map
def prepare_uncertainty_plot(uncertainty, origMask, predMask):
    ticks = [i + 0.5 for i in range(0,NUM_CLASSES+1)]
    norm = colors.BoundaryNorm([x-0.5 for x in ticks]+[NUM_CLASSES+1],NUM_CLASSES+1)
    
    fig, ax = plt.subplots(1, 3)
    
    mask_colors = mpl.colormaps['tab20'].colors
    cmap = colors.LinearSegmentedColormap.from_list("cmap", mask_colors[:NUM_CLASSES+3], N=NUM_CLASSES+1)
    
    fig, (ax1,ax2,ax3) = plt.subplots(1, 3,figsize=(6,6))
    for ax in fig.get_axes():
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
    
    orig = ax3.imshow(uncertainty,cmap = 'jet',norm = colors.Normalize(0,1))
    gt = ax2.imshow(origMask,cmap = cmap,norm=norm)
    ax1.imshow(predMask,cmap = cmap,norm=norm)
    
    ax3.set_title("Uncertainty")
    ax2.set_title("Original Mask")
    ax1.set_title("Prediction")

    plt.colorbar(orig,ax=ax3,orientation = 'vertical',fraction=0.046, pad=0.04)
    cbar = plt.colorbar(gt, ax=fig.get_axes(), orientation='horizontal',fraction=.05,ticks = ticks)
    cbar.ax.set_xticklabels(CLASS_NAMES)
    cbar.ax.xaxis.set_tick_params(rotation=90, labelsize=10)
    plt.show()

#calculate agreement entropy 
#arr is an array with class label for each forward pass, ex. [0,0,0] if class zero is selected for every of 3 forward passes
def calc_entropy_freq(arr):
    tot = len(arr)
    _, counts = np.unique(arr,return_counts = True)
    return -sum(
        [ ((i/tot)*math.log(i/tot)/math.log(NUM_CLASSES)) for i in counts]
    )
#calculate entropy from the already averaged softmax
def calc_entropy_softmax(softmax):
    try:
        E = -sum(
        [(pi)*math.log(pi)/math.log(len(softmax)) for pi in softmax if pi!=0]
        )
    except:
        print("Error!!" + str(softmax))
    return E
#reverse the params of the geometric augmentation
#only works for RandomRotate90 and SchiftScaleRotate
#albumentations offers no direct way to do this, so it has to be manually implemented for each new augmenatation
#only has to be done for geometric transofrms, to assign uncertainty to correct pixels
def reverse(params):
    for t in params['transforms']:
        if t['__class_fullname__'] == 'RandomRotate90' and t['applied'] == True:
            t['params']['factor'] = 4 - t['params']['factor'] 
        if t['__class_fullname__'] == 'ShiftScaleRotate' and t['applied'] == True:
            #reverse transform matrix the matrix 
            m = (t['params']['matrix'])

            m = np.array(m)
            inv = np.linalg.inv(m)

            t['params']['matrix'] = inv
    #returns the params to be used for the reverse transform. 
    return params

In [ ]:
#function for obtaining tta uncertainty map for a single patch
#show_plot - show uncertainty map, show_pred_plots - show predictions for each forward pass, useful for debugging, N is the number of forward passes
def tta(model_dict:ModelWrapper,id,aug_img,geo_aug,N=10,show_plot = False,show_pred_plots=False):
    model_dict.model.eval()
    #averaged softmax for every pixel
    res_softmax = np.zeros((NUM_CLASSES,INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH))
    #list of predicted class labels for every pixel
    res_prediction = np.empty((N,INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH))
    
    with torch.no_grad():
        for i in range(N):
            x,y,params = model_dict.testDataset.test_time_augment(id,aug_img,geo_aug)
            params = reverse(params)
            x = [val.to(DEVICE).unsqueeze(0) for val in x]
            predMask = model_dict.model(x).squeeze()
            
            #predMask _softmax contains the softmax for each pixel, predMask_prediction contains the class labels
            if model_dict.fusion_type != FusionType.LATE_SIMPLE:
                #in case of late simple fusion softmax has already been applied
                predMask_softmax = torch.nn.functional.softmax(input= predMask, dim=0).cpu().numpy()
            else:
                predMask_softmax = predMask.cpu().numpy()
            predMask_prediction = torch.argmax(predMask, dim=0).cpu().numpy()
            
            
            img = x[0].squeeze(0).cpu().numpy()
            #reverse the geometrical augmentation
            t = A.ReplayCompose.replay(params,image = np.transpose(img,(1,2,0)),mask = np.transpose(predMask_softmax,(1,2,0)))
            
            img = np.transpose(t['image'],(2,0,1))
            predMask_softmax = np.transpose(t['mask'],(2,0,1))
            
            y = y.cpu().numpy()
            
            t2 = A.ReplayCompose.replay(params,image = y,mask = predMask_prediction)
            y = t2['image']
            predMask_prediction = t2['mask']

            res_prediction[i] = predMask_prediction
            res_softmax += predMask_softmax
            
            if show_pred_plots:
                predMask_softmax = np.argmax(predMask_softmax,axis = 0)
                prepare_plot(img,y,predMask_softmax,img_type=model_dict.name[:3])
        #average the softmax results
        res_softmax = res_softmax/N
        
        #calculate entropy for each pixel
        entropies_softmax = np.empty((INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH))
        entropies_prediction = np.empty((INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH))
        for i in range(INPUT_IMAGE_HEIGHT):
            for j in range(INPUT_IMAGE_WIDTH):
                entropies_softmax[i][j] = (calc_entropy_softmax(res_softmax[:,i,j]))
                entropies_prediction[i][j] = (calc_entropy_freq(res_prediction[:,i,j]))

        if show_plot:
            images,m = model_dict.testDataset[id]
            prediction = model_dict.model([image.to(DEVICE).unsqueeze(0) for image in images]).squeeze()
            prediction = torch.argmax(prediction, dim=0).cpu().numpy()
            m = np.vectorize(lambda x: x if x >= 0 else 16)(m).astype(np.uint8)
            prepare_uncertainty_plot(entropies_prediction, m,prediction)
        
            prepare_uncertainty_plot(entropies_softmax, m,prediction)
    
    return {
        'avg_entropy_softmax':(entropies_softmax),
        'avg_entropy_pred':(entropies_prediction),
        'imgId':id,        
    }

#compute uncertainty using mcdo for one sample
def mcdo(model_dict:ModelWrapper,id,N=10,show_plot = False,show_pred_plots=False):
    #activate the dropout layers
    model_dict.model.eval()
    for m in model_dict.model.modules():
        if m.__class__.__name__.startswith('Dropout'):
            m.train()
            
    #averaged softmax for every pixel
    res_softmax = np.zeros((NUM_CLASSES,INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH))
    #list of predicted class labels for every pixel
    res_prediction = np.empty((N,INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH))
    
    with torch.no_grad():
        for i in range(N):
            x,y = model_dict.testDataset[id]
            
            x = [val.to(DEVICE).unsqueeze(0) for val in x]
            
            predMask = model_dict.model(x).squeeze()
            
            if model_dict.fusion_type != FusionType.LATE_SIMPLE:
                predMask_softmax = torch.nn.functional.softmax(input= predMask, dim=0).cpu().numpy()
            else:
                predMask_softmax = predMask.cpu().numpy()   
            predMask_prediction = torch.argmax(predMask, dim=0).cpu().numpy()
            
            
            img = x[0].squeeze(0).cpu().numpy()
            
            y = y.cpu().numpy()

            res_prediction[i] = predMask_prediction
            res_softmax += predMask_softmax
            
            if show_pred_plots:
                predMask_softmax = np.argmax(predMask_softmax,axis = 0).transpose((1,2,0))
                prepare_plot(img,y,predMask_softmax,img_type=model_dict.name[:3])
        #average softmax
        res_softmax = res_softmax/N
        #compute entropy
        entropies_softmax = np.empty((INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH))
        entropies_prediction = np.empty((INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH))

        for i in range(INPUT_IMAGE_HEIGHT):
            for j in range(INPUT_IMAGE_WIDTH):
                entropies_softmax[i][j] = (calc_entropy_softmax(res_softmax[:,i,j]))
                entropies_prediction[i][j] = (calc_entropy_freq(res_prediction[:,i,j]))

        
        if show_plot:
            images,m = model_dict.testDataset[id]
            prediction = model_dict.model([image.to(DEVICE).unsqueeze(0) for image in images]).squeeze()
            prediction = torch.argmax(prediction, dim=0).cpu().numpy()
            m = np.vectorize(lambda x: x if x >= 0 else 16)(m).astype(np.uint8)
        
            prepare_uncertainty_plot(entropies_prediction, m,prediction)
            prepare_uncertainty_plot(entropies_softmax, m,prediction)
    
    return {
        'entropy_softmax':entropies_softmax,
        'entropy_pred':(entropies_prediction),
        'imgId':id,
        
    }

#get and save entropy for all samples
def get_entropy_metrics(model_wrapper,estimation_method,images_aug=None,geo_aug=None,N=30):
    entropies_predictive = [[] for _ in range(len(model_wrapper.testDataset))]
    entropies_agreement = [[] for _ in range(len(model_wrapper.testDataset))]
    
    for i in range(len(model_wrapper.testDataset)):
        if estimation_method == 'mcdo':
            results = mcdo(model_wrapper,i,N=N)
        elif estimation_method == "tta":
            results = tta(model_wrapper,i,images_aug,geo_aug,N)
        entropies_agreement[i] = (results['entropy_pred'])
        entropies_predictive[i] = (results['entropy_softmax'])
    model_wrapper.uncertainty_dict[estimation_method]["uncertainties_pred"] = entropies_agreement
    model_wrapper.uncertainty_dict[estimation_method]["uncertainties_soft"] = entropies_predictive


In [ ]:
#get all uncertainties for all models with a given fusion method
def get_all_uncertainties(models,fusion_type):
    #gaussian noise, parameters chosen experimentally
    image_only_aug = [
                  A.GaussNoise(p=1,  mean=0.0,var_limit=(0.000005,0.00001),per_channel=True)
                 ]
    #vertical shift, border_mode = 3 to include all pixels
    geo_aug = [A.ShiftScaleRotate(p=1,scale_limit=0,rotate_limit=0,shift_limit_x = 0,shift_limit_y=(-0.25,0.25),border_mode=3,interpolation=0)]

    for model_key in models[fusion_type].keys():
        print("Fetching uncertainties for " + str(fusion_type) + " " + model_key + '...')
        model = models[fusion_type][model_key]
        get_entropy_metrics(model,estimation_method="tta",images_aug=image_only_aug,geo_aug=geo_aug,N=10)

def get_all_uncertainties_mcdo(models,fusion_type):
    for model_key in models[fusion_type].keys():
        print("Fetching uncertainties for " + str(fusion_type) + " " + model_key + '...')
        model = models[fusion_type][model_key]
        get_entropy_metrics(model,estimation_method='mcdo',N=10)


In [ ]:
#get average uncertainty for each class (pixels, where this class is correct)
def get_class_uncs(model:ModelWrapper):
    res_pred = [[] for cls in CLASS_NAMES]
    res_soft = [[] for cls in CLASS_NAMES]
    for (x,(_,mask)) in enumerate(models[FusionType.NO_FUSION]['sar'].testDataset):
        mask = mask.numpy()
        for i in range(128):
            for j in range(128):
                cls_id = mask[i][j]
                if cls_id == len(res_pred):
                    cls_id = -1
                res_pred[cls_id].append(model.uncertainty_dict['uncertainties_pred'][x][i][j])
                res_soft[cls_id].append(model.uncertainty_dict['uncertainties_soft'][x][i][j])
    res_soft = [(np.mean(unc)) for unc in res_soft]
    res_pred = [(np.mean(unc)) for unc in res_pred]
    return (res_soft, res_pred)


# Graphics for analysis

In [ ]:
models[FusionType.MIDDLE]['hsi_lms_rgb'].metrics = {"OA":0.65,"AA":0.59,"mIOU":0.46,"k":0.62}
models[FusionType.MIDDLE]['lms_rgb'].metrics = {"OA":0.64,"AA":0.57,"mIOU":0.44,"k":0.62}
models[FusionType.LATE]['hsi_lms'].metrics = {"OA":0.63,"AA":0.55,"mIOU":0.43,"k":0.61}
models[FusionType.LATE]['hsi_lms_rgb'].metrics = {"OA":0.62,"AA":0.55,"mIOU":0.42,"k":0.59}
models[FusionType.MIDDLE]['hsi_rgb'].metrics = {"OA":0.61,"AA":0.55,"mIOU":0.42,"k":0.58}
models[FusionType.EARLY]['hsi_lms_rgb'].metrics = {"OA":0.62,"AA":0.54,"mIOU":0.40,"k":0.60}
models[FusionType.LATE]['lms_rgb'].metrics = {"OA":0.61,"AA":0.54,"mIOU":0.41,"k":0.58}
models[FusionType.MIDDLE]['hsi_lms'].metrics = {"OA":0.59,"AA":0.51,"mIOU":0.39,"k":0.57}
models[FusionType.EARLY]['lms_rgb'].metrics = {"OA":0.59,"AA":0.52,"mIOU":0.40,"k":0.56}
models[FusionType.NO_FUSION]['lms'].metrics = {"OA":0.62,"AA":0.54,"mIOU":0.41,"k":0.59}
models[FusionType.EARLY]['hsi_lms'].metrics = {"OA":0.59,"AA":0.50,"mIOU":0.37,"k":0.56}
models[FusionType.LATE]['hsi_rgb'].metrics = {"OA":0.55,"AA":0.49,"mIOU":0.35,"k":0.52}
models[FusionType.EARLY]['hsi_rgb'].metrics = {"OA":0.54,"AA":0.48,"mIOU":0.34,"k":0.51}
models[FusionType.NO_FUSION]['rgb'].metrics = {"OA":0.53,"AA":0.47,"mIOU":0.33,"k":0.50}
models[FusionType.NO_FUSION]['hsi'].metrics = {"OA":0.54,"AA":0.46,"mIOU":0.33,"k":0.51}


In [ ]:
#show a given metric in a bar plot
def get_bar_plot_acc(models,metric = 'AA',title = "",save = False, save_path = ""):
    
    single_mods = [m.upper() for m in models[FusionType.NO_FUSION]]
    modalities = [m.replace("_",",").upper() for m in models[FusionType.EARLY]]
    x1 = np.arange(len(single_mods))
    x2 = np.arange(0,len(modalities)*2,2) + len(x1)  # the label locations

    width = 0.4 # the width of the bars
    multiplier = 0
    _, ax = plt.subplots(layout='constrained',figsize = (8,6))
    ax.bar(x1,[np.mean(models[FusionType.NO_FUSION][m].metrics[metric]) for m in models[FusionType.NO_FUSION]],width,label = "NO FUSION")
    
    for f in models:
        if f == FusionType.NO_FUSION:
            continue      
        offset = width * multiplier
        ax.bar(x2 + offset, [np.mean(models[f][m].metrics[metric]) for m in models[f]], width, label=str(f).split('.')[1].replace("_"," ")+" FUSION")

        multiplier += 1

    # Add some text for labels, title and custom x-axis tick labels, etc.

    ax.set_xlabel('Combination of modalities',fontsize=18)
    ax.set_xticks(np.concatenate(((x1),(x2 + width))), single_mods + modalities)
    plt.xticks(rotation = 90)
    ax.legend(loc='upper left', ncols=1,bbox_to_anchor=(1, 0.5))

    ax.set_title(title,fontsize=18)
    if save:
        plt.savefig(f"{save_path}/{title}.pdf")
    plt.show()

In [ ]:
get_bar_plot_acc(models,metric="OA",title="Overall accuracy",save = True,save_path="results/barplots" )
get_bar_plot_acc(models,metric="AA",title="Average accuracy",save = True,save_path="results/barplots" )
get_bar_plot_acc(models,metric="mIOU",title="Mean intersection over union",save = True,save_path="results/barplots" )
get_bar_plot_acc(models,metric="k",title="Cohen's kappa",save = True,save_path="results/barplots" )

In [ ]:
#show average uncertainty in a bar plot, type1 = mcdo/tta, type2 = uncertainties_pred/uncertainties_soft
def get_bar_plot(models,type1 = 'tta',type2 = 'soft',title = "",save = False, save_path = ""):
    single_mods = [m.upper() for m in models[FusionType.NO_FUSION]]
    modalities = [m.replace("_",",").upper() for m in models[FusionType.EARLY]]
    x1 = np.arange(len(single_mods))
    x2 = np.arange(0,len(modalities)*2,2) + len(x1)  # the label locations
    print(x2)
    width = 0.4 # the width of the bars
    multiplier = 0
    fig, ax = plt.subplots(layout='constrained',figsize = (8,6))

    ax.bar(x1,[np.mean(models[FusionType.NO_FUSION][m].uncertainty_dict[type1]['uncertainties_'+type2]) for m in models[FusionType.NO_FUSION]],width,label = "NO FUSION")
    
    for f in models:
        if f == FusionType.NO_FUSION:
            continue      
        offset = width * multiplier
        ax.bar(x2 + offset, [np.mean(models[f][m].uncertainty_dict[type1]['uncertainties_'+type2]) for m in models[f]], width, label=str(f).split('.')[1].replace("_"," ")+" FUSION")
        #ax.bar_label(rects, padding=3)
        multiplier += 1

    # Add some text for labels, title and custom x-axis tick labels, etc.
    ax.set_ylabel('Average uncertainty',fontsize=18)
    ax.set_xlabel('Combination of modalities',fontsize=18)
    ax.set_xticks(np.concatenate(((x1),(x2 + width))), single_mods + modalities)
    plt.xticks(rotation = 90)
    ax.legend(loc='upper left', ncols=1,bbox_to_anchor=(1, 0.5))

    ax.set_title(title,fontsize=18)
    if save:
        plt.savefig(f"{save_path}/{title}.pdf")
    return fig

In [ ]:
get_bar_plot(models,type1='mcdo',type2='pred',title="Average epistemic agreement uncertainty",save = True,save_path="results/barplots" )
get_bar_plot(models,type1='mcdo',type2='soft',title="Average epistemic predictive uncertainty",save = True,save_path="results/barplots" )
get_bar_plot(models,type1='tta',type2='pred',title="Average aleatoric agreement uncertainty",save = True,save_path="results/barplots" )
get_bar_plot(models,type1='tta',type2='soft',title="Average aleatoric predictive uncertainty",save = True,save_path="results/barplots" )

In [ ]:
from matplotlib.lines import Line2D
def  get_scattered_acc_unc(models,type1 = 'tta',type2 = 'soft',acc_type = "OA",title = "",save = False, save_path = ""):
    shapes = {
        FusionType.NO_FUSION:'o',
        FusionType.EARLY:'D',
        FusionType.MIDDLE: "^",
        FusionType.LATE:"s"
    }

   
    colors = {
        'hsi':'darkorange',
        'lms':'green',
        'rgb':'aqua',
        'hsi_rgb':'orangered',
        'lms_rgb':'burlywood',
        'hsi_lms':'lime',
        'hsi_lms_rgb':'blue',
        'hsi_dsm_msi':'fuchsia',
        'hsi_sar_dsm_msi':'blue'

    }
    
    
    fig,ax = plt.subplots(layout = 'constrained',figsize = (9,6))
    for fusion in models:
        ax.scatter([models[fusion][m].metrics[acc_type] for m in models[fusion]],[np.mean(models[fusion][m].uncertainty_dict[type1]["uncertainties_" + type2]) for m in models[fusion]],c=[colors[m]for m in models[fusion]],marker=shapes[fusion], s=200,label=[str(fusion).split('.')[1]+" FUSION " + m for m in models[fusion]])

  
    fake_colors = [Line2D([0], [0], marker='o', color = 'w',markerfacecolor=colors[m], label=m.upper(),markersize=12)
                   for m in models[FusionType.NO_FUSION]
                   ] + [Line2D([0], [0], marker='o', color = 'w',markerfacecolor=colors[m], label=m.replace("_",",").upper(),markersize=12)
                   for m in models[FusionType.EARLY]
                   ]
   
    fake_shapes = [Line2D([0], [0], marker=shapes[f], color='w',markerfacecolor='black', label=str(f).split(".")[1].replace("_"," "),markersize=15)
                   for f in shapes.keys()
                   ]
    ax.legend(handles=fake_colors)
    ax.set_ylabel('Average uncertainty',fontsize=18)
    xlabel = "Overall accuracy" if acc_type == "OA" else "Average accuracy"
    ax.set_xlabel(xlabel,fontsize=18)
    l1= ax.legend(handles=fake_colors,loc = 'upper left',bbox_to_anchor = (1,0.35))
    ax.add_artist(l1)
    ax.legend(handles=fake_shapes,loc = 'upper left',bbox_to_anchor = (1,0.75))

  
    #ax.set_xlim(ax.get_ylim())
    ax.set_title(title,fontsize=18)
    print(ax.get_legend_handles_labels())
    if save:
        plt.savefig(f"{save_path}/{title}.pdf")
    plt.show()
    
    plt.close()

In [ ]:
get_scattered_acc_unc(models,type1 = 'tta',type2 = 'soft',acc_type = "AA",title = "Aleatoric predictive uncertainty vs. average accuracy",save = True, save_path = "results/scatterplots")
get_scattered_acc_unc(models,type1 = 'tta',type2 = 'pred',acc_type = "AA",title = "Aleatoric agreement uncertainty vs. average accuracy",save = True, save_path = "results/scatterplots")
get_scattered_acc_unc(models,type1 = 'mcdo',type2 = 'soft',acc_type = "AA",title = "Epistemic predictive uncertainty vs. average accuracy",save =True, save_path = "results/scatterplots")
get_scattered_acc_unc(models,type1 = 'mcdo',type2 = 'pred',acc_type = "AA",title = "Epistemic agreement uncertainty vs. average accuracy",save = True, save_path = "results/scatterplots")

In [ ]:
from matplotlib.lines import Line2D
from cycler import cycler
def  get_scattered_unc_acc_change(models,type1 = 'tta',type2 = 'soft',acc_type = "OA",title = "",save = False, save_path = "",unc_comp_method = min,acc_comp_method = max):
    shapes = {
        FusionType.NO_FUSION:'o',
        FusionType.EARLY:'D',
        FusionType.MIDDLE: "^",
        FusionType.LATE:"s"
    }

   
    colors = {
        'hsi':'darkorange',
        'lms':'green',
        'rgb':'aqua',
        'hsi_rgb':'orangered',
        'lms_rgb':'burlywood',
        'hsi_lms':'lime',
        'hsi_lms_rgb':'blue',
        'hsi_dsm_msi':'fuchsia',
        'hsi_sar_dsm_msi':'red'

    }
    unc_change = {f:{m:np.mean(models[f][m].uncertainty_dict[type1]["uncertainties_"+type2] - unc_comp_method([np.mean(models[FusionType.NO_FUSION][mod1].uncertainty_dict[type1]["uncertainties_"+type2]) for mod1 in m.split("_")])) for m in models[f]} for f in models if f != FusionType.NO_FUSION}
    acc_change = {f:{m:models[f][m].metrics[acc_type] - acc_comp_method([models[FusionType.NO_FUSION][mod1].metrics[acc_type] for mod1 in m.split("_")]) for m in models[f]} for f in models if f != FusionType.NO_FUSION}

    _,ax = plt.subplots(layout = 'constrained',figsize = (8.5,6))
    for fusion in models:
        
        if fusion == FusionType.NO_FUSION:
            continue
        plot = ax.scatter([acc_change[fusion][m] for m in models[fusion]],[unc_change[fusion][m] for m in models[fusion]],c=[colors[m]for m in models[fusion]],s=200,marker=shapes[fusion], label=[str(fusion).split('.')[1]+" FUSION " + m for m in models[fusion]])

  
    fake_colors = [Line2D([0], [0], marker='o', color = 'w',markerfacecolor=colors[m], label=m.replace("_",",").upper(),markersize=10)
                   for m in models[FusionType.EARLY]
                   ]
   
    fake_shapes = [Line2D([0], [0], marker=shapes[f], color='w',markerfacecolor='black', label=str(f).split(".")[1].replace("_"," "),markersize=10)
                   for f in shapes.keys()
                   ]
    
    
    xlabel = "overall accuracy" if acc_type == "OA" else "average accuracy"
    ax.set_ylabel('Change in average uncertainty',fontsize=18)
    ax.set_xlabel("Change in "+ xlabel,fontsize=18)
    
    
    l1= ax.legend(handles=fake_shapes,loc = 'upper left',bbox_to_anchor = (1,0.35))
    ax.add_artist(l1)
    ax.legend(handles=fake_colors,loc = 'upper left',bbox_to_anchor = (1,0.75))

  
    #ax.set_xlim(ax.get_ylim())
    ax.set_title(title,fontsize=18)
    title = title.replace("\n","")

    if save:
        plt.savefig(f"{save_path}/{title}.pdf")
    plt.show()
    
    plt.close()

In [ ]:
get_scattered_unc_acc_change(models,type1 = 'tta',type2 = 'soft',acc_type = "AA",save_path = "results/scatterplots",save = True, title = "Change in average aleatoric predictive uncertainty\n vs change in average accuracy",unc_comp_method = np.min,acc_comp_method = max)
get_scattered_unc_acc_change(models,type1 = 'tta',type2 = 'pred',acc_type = "AA",save_path = "results/scatterplots",save = True, title = "Change in average aleatoric agreement uncertainty\n vs change in average accuracy",unc_comp_method = np.min,acc_comp_method = max)
get_scattered_unc_acc_change(models,type1 = 'mcdo',type2 = 'soft',acc_type = "AA",save_path = "results/scatterplots",save = True,title = "Change in average epistemic predictive uncertainty\n vs change in average accuracy",unc_comp_method = np.min,acc_comp_method = max)
get_scattered_unc_acc_change(models,type1 = 'mcdo',type2 = 'pred',acc_type = "AA",save_path= "results/scatterplots",save = True,title = "Change in average epistemic agreement uncertainty\n vs change in average accuracy",unc_comp_method = np.min,acc_comp_method = max)

In [ ]:
#get ECE and reliability diagram
def get_calibration_metrics(model,type1 = 'tta', type2 = 'soft', save = False,save_folder = '',show_plot = False):
    n_bins = 10
    
    
    preds = np.load(f"{PREDICTIONS_PATH}{str(model.fusion_type).split('.')[1]}_{model.name}_predictions.pth.npy",allow_pickle = True).item()
    masks = np.array(preds['masks']).flatten()

    preds = np.array(preds['preds']).flatten()

    flat_uncs = model.uncertainty_dict[type1]['uncertainties_'+type2].flatten()

    min = np.min(flat_uncs)
    maxi = np.max(flat_uncs)
    #scale to 0-1, important for agreement uncertainty
    flat_uncs = [(x-min)/(maxi-min) for x in flat_uncs]
    step = (np.max(flat_uncs) - np.min(flat_uncs))/n_bins
    res = []
    curr = np.min(flat_uncs)
    #total number of samples in each bin
    totals = [0 for _ in range(n_bins)]
    #uncertainties in each bin, used for average uncertainty to calculate ece
    uncs = [[] for _ in range(n_bins)]
    

    for i in range(n_bins):
        total = 0
        truth = 0
        for j in range(len(flat_uncs)):
            if flat_uncs[j] >= curr and flat_uncs[j]<curr+step:
                if masks[j] != NUM_CLASSES:
                    total += 1
                    truth += masks[j] == preds[j]
                    uncs[i].append(flat_uncs[j])

        totals[i] = (total)
        curr += step
        if total != 0:
            res.append(truth/total)
        else:
            res.append(0)
        
    uncs = [sum(uncs[i])/totals[i] if totals[i] != 0 else 0 for i in range(n_bins)]
    #res, conf and totals have to reversed for correct order of bars and correct ece calculation
    res.reverse()   
    xs = [(2*np.min(flat_uncs) + step*i+step*(i+1))/2 for i in range(n_bins)]
    conf = [1-unc for unc in uncs]
    conf.reverse()
    
    plt.figure()
    plt.bar(xs,res,width=1/n_bins,color='blue')
    plt.plot([0,1],[0,1],color = 'red',label = "Unceratainty")
    plt.xlabel("Confidence")
    plt.ylabel('Accuracy')
    plt.grid(True)
    
    totals.reverse()
    ece = sum([totals[i]* (abs(res[i] - conf[i]))/sum(totals)  for i in range(n_bins) ])
    
    props = dict(boxstyle='round', facecolor='white', alpha=0.5)   
    textstr = f"ECE = {ece:.4f}"
    
    plt.text(0.05, 0.95, textstr,  fontsize=14,
        verticalalignment='top', bbox=props)
    
    if show_plot:
        plt.show()
    if save:
        plt.savefig(BASE_OUTPUT + f"{save_folder}/{str(model.fusion_type).split('.')[1]}_{model.name}_calibration.pdf")
    plt.close('all')
    
    
    return ece

In [ ]:
#similar to the calibration function, but returns the reliability line to plot multiple lines in one figure
def get_reliability_line(model,type1 = 'tta', type2 = 'soft', save = False,save_folder = '',show_plot = False):
    n_bins = 10
    preds = np.load(f"{PREDICTIONS_PATH}{str(model.fusion_type).split('.')[1]}_{model.name}_predictions.pth.npy",allow_pickle = True).item()
    
    masks = np.array(preds['masks']).flatten()
    
    preds = np.array(preds['preds']).flatten()
    flat_soft = model.uncertainty_dict[type1]['uncertainties_'+type2].flatten()
    min = np.min(flat_soft)
    maxi = np.max(flat_soft)
    flat_soft = [(x-min)/(maxi-min) for x in flat_soft]
    step = (np.max(flat_soft) - np.min(flat_soft))/n_bins
    res = []
    curr = np.min(flat_soft)
    totals = [0 for _ in range(n_bins)]
    uncs = [[] for _ in range(n_bins)]
    

    for i in range(n_bins):
        total = 0
        truth = 0
        for j in range(len(flat_soft)):
            if flat_soft[j] >= curr and flat_soft[j]<curr+step:
                if masks[j] != NUM_CLASSES:
                    total += 1
                    truth += masks[j] == preds[j]
                    uncs[i].append(flat_soft[j])

        totals[i] = (total)
        curr += step
        if total ==0:
            res.append(0)
        else:
            res.append(truth/total)

    res.reverse()   
    
    return res



In [ ]:
#get all unimodal and multimodal reliabilty lines for a given combination
def get_lines(models,combo):
    tta_pred = []
    tta_soft = []
    mcdo_pred = []
    mcdo_soft = []
    for x in combo:
        m=models[FusionType.NO_FUSION][x]
        tta_pred.append(get_reliability_line(m,save=False,type1='tta',type2="pred"))
        tta_soft.append(get_reliability_line(m,save=False,type1='tta',type2="soft"))
        mcdo_pred.append( get_reliability_line(m,save=False,type1='mcdo',type2="pred"))
        mcdo_soft.append( get_reliability_line(m,save=False,type1='mcdo',type2="soft"))
    for f in [FusionType.EARLY, FusionType.MIDDLE,FusionType.LATE]:
        m = models[f]["_".join(combo)]
        tta_pred.append(get_reliability_line(m,save=False,type1='tta',type2="pred"))
        tta_soft.append(get_reliability_line(m,save=False,type1='tta',type2="soft"))
        mcdo_pred.append( get_reliability_line(m,save=False,type1='mcdo',type2="pred"))
        mcdo_soft.append( get_reliability_line(m,save=False,type1='mcdo',type2="soft"))
    return [tta_pred,tta_soft,mcdo_pred,mcdo_soft]
    
            
   

In [ ]:
#draw and save the diagrams
def rel_diag_line(res,combo,save=False,name="",tit_string =""):
    fs = [FusionType.EARLY,FusionType.MIDDLE,FusionType.LATE]
    for j in range(len(res)):
            colors = mpl.colormaps['tab10'].colors
            label = combo[j] if j <len(combo) else str(fs[j-len(combo)]).split(".")[1] + " FUSION"
            plt.plot([0.1*i for i in range(11)],[0]+res[j],color=colors[j],label=label.upper())
           
    plt.xlabel("Confidence")
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True) 
    plt.plot([0,1],[0,1],"--",color = 'black',label = "")  
    plt.title("Reliability diagram computed with\n" +tit_string)
    if save:
        plt.savefig(BASE_OUTPUT + f"results/{'_'.join(combo)}_{name}_calibration.pdf")
    plt.show()

# Example

In [ ]:
#y start and stop
x1 = random.randint(0,601-149)
x1 =216
x2 = x1 +149


#x start and stop
x3 = 0 
x4 =149


names = ['hsi','lms','rgb']
fs = [FusionType.EARLY,FusionType.MIDDLE,FusionType.LATE]
ms = [models[FusionType.NO_FUSION][mod] for mod in names]+ [models[f]["_".join(names)] for f in fs]

#id of example patch
x = random.randint(0,len(ms[0].testDataset))
x=77

#segmentation maps
preds = [m.preds['preds'][x] for m in ms]

#visualisation of data
mods = [ms[i].testDataset[x][0][0].numpy().transpose((1,2,0)) for i in range(len(names))]
mods = [m[x1:x2,x3:x4,:] for m in mods]
if 'hsi' in names:
    mods[names.index('hsi')] = show_hsi( mods[names.index('hsi')])
#crop the preds
preds = [m[x1:x2,x3:x4] for m in preds]

rows = len(ms) + 1

gt = ms[0].testDataset[x][1].numpy()
gt = gt + (22*(gt == -1))
gt = gt[x1:x2,x3:x4]

fig, axs = plt.subplots(rows, 6,figsize=(25,20),layout = 'constrained')

for ax in fig.get_axes():
    
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_ticks([])
    

for i in range(len(names)):
    axs[i+1][0].set_ylabel(names[i].upper(),fontsize='large')
    
axs[len(names)+1][0].set_ylabel("Early Fusion",fontsize='large')
axs[len(names)+2][0].set_ylabel("Middle Fusion",fontsize='large')
axs[len(names)+3][0].set_ylabel("Late Fusion",fontsize='large')
ticks = [i+0.5 for i in range(NUM_CLASSES+1)]

norm = colors.BoundaryNorm([x-0.51 for x in ticks]+[21],ncolors = NUM_CLASSES+1)

mask_colors = mpl.colormaps['tab20'].colors
cmap = colors.LinearSegmentedColormap.from_list("cmap", mask_colors[:22], N=NUM_CLASSES+1)

gt_ax = axs[0][0].imshow(gt,cmap = cmap,norm=norm,interpolation='none')
for i in range(len(mods)):
    axs[0][i+1].imshow(mods[i])

for i in range(len(ms)):
    #pixels where prediction != ground truth and ground truth != unclassified
    error_map = (gt != preds[i]) ^ (gt == 21)
    axs[i+1][1-1].imshow(preds[i],cmap = cmap,norm=norm,interpolation='none')

    axs[i+1][2-1].imshow(ms[i].uncertainty_dict['tta']['uncertainties_soft'][x][x1:x2,x3:x4],cmap = 'jet',norm = colors.Normalize(0,1))
    axs[i+1][3-1].imshow(ms[i].uncertainty_dict['tta']['uncertainties_pred'][x][x1:x2,x3:x4],cmap = 'jet',norm = colors.Normalize(0,1))
    axs[i+1][4-1].imshow(ms[i].uncertainty_dict['mcdo']['uncertainties_soft'][x][x1:x2,x3:x4],cmap = 'jet',norm = colors.Normalize(0,1))
    orig=axs[i+1][5-1].imshow(ms[i].uncertainty_dict['mcdo']['uncertainties_pred'][x][x1:x2,x3:x4],cmap = 'jet',norm = colors.Normalize(0,1))
    err = axs[i+1][6-1].imshow(error_map,cmap='jet',interpolation = 'nearest')
    
plt.savefig("")
plt.show()

In [ ]:


#generate the two colorbars (classes and uncertainty)
norm = colors.BoundaryNorm([x-0.5 for x in ticks]+[21],NUM_CLASSES+1)
mask_colors = mpl.colormaps['tab20'].colors
cmap = colors.LinearSegmentedColormap.from_list("cmap", mask_colors[:22], N=NUM_CLASSES+1)

colorbars, cbar_axs = plt.subplots(1,2,figsize=(25,0.75),facecolor=(1, 1, 1))

cbar = colorbars.colorbar(gt_ax, cax=cbar_axs[0], orientation='horizontal',ticks = ticks)
cbar.ax.set_xticklabels(CLASS_NAMES)
cbar.ax.xaxis.set_tick_params(rotation=90)
colorbars.colorbar(orig,orientation = 'horizontal',cax=cbar_axs[1])
plt.subplots_adjust(bottom=0.15)
colorbars.savefig("", bbox_inches="tight")